# Text FTM Attack Runner (Colab + VS Code)

This notebook runs your full experiment on 100 samples with surrogate + black-box evaluation and saves results.

## 1) Locate Project Directory

If running in Colab, make sure your project folder exists under one of:
- `/content/FTM_text/text_ftm_attack`
- `/content/drive/MyDrive/FTM_text/text_ftm_attack`

If running in VS Code locally, it will use the current folder automatically.

In [3]:
# Run this cell first in Colab.
# It mounts Drive and sets the project location for the next cell.
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print('Drive mount skipped (not in Colab or already mounted):', e)

# Update this path if your folder is different in Drive.
candidate = Path('/content/drive/MyDrive/FTM_text/text_ftm_attack')

if candidate.exists():
    os.environ['FTM_PROJECT_DIR'] = str(candidate)
    print('FTM_PROJECT_DIR set to:', candidate)
else:
    print('Project path not found at:', candidate)
    print('Upload/copy the folder to Drive, then set:')
    print("os.environ['FTM_PROJECT_DIR'] = '/content/drive/MyDrive/<your-path>/text_ftm_attack'")

Drive mount skipped (not in Colab or already mounted): User cancelled dfs_ephemeral authorization
Project path not found at: /content/drive/MyDrive/FTM_text/text_ftm_attack
Upload/copy the folder to Drive, then set:
os.environ['FTM_PROJECT_DIR'] = '/content/drive/MyDrive/<your-path>/text_ftm_attack'


In [4]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime

def is_project_dir(p: Path) -> bool:
    return (p / 'main.py').exists() and (p / 'requirements.txt').exists()

def find_project_dir():
    # Optional manual override for any environment.
    env_path = os.environ.get('FTM_PROJECT_DIR')
    if env_path:
        p = Path(env_path).expanduser()
        if is_project_dir(p):
            return p.resolve()

    # Fast checks for common local/Colab layouts.
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/FTM_text/text_ftm_attack'),
        Path('/content/drive/MyDrive/FTM_text/text_ftm_attack'),
    ]
    for p in candidates:
        if is_project_dir(p):
            return p.resolve()

    # Recursive fallback search under common roots.
    search_roots = [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive')]
    for root in search_roots:
        if not root.exists():
            continue
        for main_file in root.rglob('main.py'):
            p = main_file.parent
            if is_project_dir(p):
                return p.resolve()

    return None

PROJECT_DIR = find_project_dir()
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find text_ftm_attack folder. Set FTM_PROJECT_DIR or place project in /content/... and rerun."
    )

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
print('Python =', sys.executable)

FileNotFoundError: Could not find text_ftm_attack folder. Set FTM_PROJECT_DIR or place project in /content/... and rerun.

## 2) Check GPU

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    DEVICE = 'cuda:0'
else:
    DEVICE = 'cpu'
print('DEVICE =', DEVICE)

## 3) Install Dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Dependencies installed.')

## 4) Verify Dataset

In [ ]:
DATA_CSV = PROJECT_DIR / 'data' / 'IMDB Dataset.csv'
if not DATA_CSV.exists():
    raise FileNotFoundError(f'Missing dataset file: {DATA_CSV}')
print('Dataset found:', DATA_CSV)

## 5) Run Attack (100 Samples + Black-Box Eval)

In [ ]:
RUN_TAG = datetime.now().strftime('run_gpu_%Y%m%d_%H%M%S')
SAVE_DIR = PROJECT_DIR / 'results' / RUN_TAG
SAVE_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, 'main.py',
    '--device', DEVICE,
    '--num_samples', '100',
    '--eval',
    '--data_csv', str(DATA_CSV),
    '--save_dir', str(SAVE_DIR),
]

print('Running command:')
print(' '.join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Attack run failed with exit code {ret}')

print('Run completed.')
print('SAVE_DIR =', SAVE_DIR)

## 6) Load and Print Final Metrics

In [ ]:
summary_path = SAVE_DIR / 'results_summary.json'
examples_path = SAVE_DIR / 'examples.txt'

if not summary_path.exists():
    raise FileNotFoundError(f'Summary not found: {summary_path}')

with open(summary_path, 'r', encoding='utf-8') as f:
    summary = json.load(f)

print('===== FINAL METRICS =====')
print('num_samples:', summary.get('num_samples'))
print('surrogate_asr:', summary.get('surrogate_asr'))
print('avg_semantic_similarity:', summary.get('avg_semantic_similarity'))
print('above_sim_threshold:', summary.get('above_sim_threshold'))

bb = summary.get('black_box_asr', {})
if bb:
    print('black_box_asr:')
    for k, v in bb.items():
        print(' -', k, ':', v)
    print('avg_black_box_asr:', summary.get('avg_black_box_asr'))

print('summary_path =', summary_path)
print('examples_path =', examples_path)

## 7) Optional: Zip Results for Download

In [ ]:
import shutil
zip_path = str(SAVE_DIR) + '.zip'
shutil.make_archive(str(SAVE_DIR), 'zip', root_dir=str(SAVE_DIR))
print('Created:', zip_path)